# EXP-04: BPTI–Trypsin ΔG_bind — DRY RUN
**Quick validation run (~5-10 min) to verify notebook works end-to-end**

> Uses minimal simulation parameters. Results are NOT physically meaningful.
> Output stored separately in `v3_gpu_results/EXP-04-dry-run/`.

- **PDB:** 2PTC (1.9 Å co-crystal)
- **Chains:** E = trypsin, I = BPTI
- **Pipeline:** Prep → Minimize → NVT (10ps) → NPT (10ps) → Production (10ps) → SMD (2 reps) → US (5 windows) → WHAM → ΔG
- **Expected:** ΔG_bind = −18.0 kcal/mol
- **PASS criterion:** [−24.7, −11.3] kcal/mol

**Prerequisites:** Upload `src/` directory to Google Drive at `MyDrive/spink7_pipeline/src/`

In [1]:
# Install all dependencies via pip (OpenMM 8.x on PyPI includes CUDA 12 support)
# Step 1: Install OpenMM and core scientific packages
!pip install openmm mdtraj 'pymbar==4.0.3' mdanalysis
# Step 2: openmmtools has no cp312 wheel on PyPI — install from source
!pip install netCDF4 git+https://github.com/choderalab/openmmtools.git
# mpiplus stub — not on PyPI, GitHub build broken on cp312; only needed for MPI parallelism
import os as _os, sys as _sys
_sp = '/usr/local/lib/python3.12/dist-packages/mpiplus'
_os.makedirs(_sp, exist_ok=True)
with open(_os.path.join(_sp, '__init__.py'), 'w') as _f:
    _f.write(
        'class delayed_termination:\n'
        '    def __init__(self, func=None):\n'
        '        self._func = func\n'
        '    def __enter__(self): return self\n'
        '    def __exit__(self, *a): pass\n'
        '    def __call__(self, *args, **kwargs):\n'
        '        if self._func is not None: return self._func(*args, **kwargs)\n'
        '        if len(args) == 1 and callable(args[0]): return args[0]\n'
        '        return self\n'
        'def get_mpicomm(): return None\n'
        'def run_single_node(rank=0, broadcast_result=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def on_single_node(rank=0, broadcast_result=False, sync_nodes=False):\n'
        '    def decorator(func): return func\n'
        '    return decorator\n'
        'def distribute(func, seq, *args, send_results_to=None, sync_nodes=False, group_nodes=None):\n'
        '    results = [func(item, *args) for item in seq]\n'
        '    return list(zip(*results)) if results and isinstance(results[0], tuple) else (results, list(seq))\n'
    )
for _k in list(_sys.modules):
    if 'mpiplus' in _k:
        del _sys.modules[_k]
!pip install -q scipy matplotlib numpy pandas requests gemmi

# Verify critical imports
import importlib
all_ok = True
for pkg in ['openmm', 'openmmtools', 'mdtraj', 'pymbar', 'MDAnalysis']:
    try:
        m = importlib.import_module(pkg)
        print(f"\u2713 {pkg} {getattr(m, '__version__', '')}")
    except ImportError as e:
        print(f"\u2717 {pkg}: {e}")
        all_ok = False

if all_ok:
    print("\n\u2705 All packages installed successfully!")
else:
    raise RuntimeError("Package installation failed \u2014 check error messages above")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 143.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 143.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 156.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 152.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 5.3 MB/s eta 0:00:00
  Cloning https://github.com/choderalab/openmmtools.git to /tmp/pip-req-build-edfvkkxl
  Running command git clone --filter=blob:none --quiet https://github.com/choderalab/openmmtools.git /tmp/pip-req-build-edfvkkxl
  Resolved https://github.com/choderalab/openmmtools.git to commit ba4031da49981fe9e8b0bcec010ab53e18a03ba2
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/1

****** PyMBAR will use 64-bit JAX! *******
* JAX is currently set to 32-bit bitsize *
* which is its default.                  *
*                                        *
* PyMBAR requires 64-bit mode and WILL   *
* enable JAX's 64-bit mode when called.  *
*                                        *
* This MAY cause problems with other     *
* Uses of JAX in the same code.          *
******************************************



✓ openmmtools 0.26.0
✓ mdtraj 1.11.1.post1
✓ pymbar 4.0.3
✓ MDAnalysis 2.10.0

✅ All packages installed successfully!


In [3]:
# Cell 2: Mount Google Drive and set up paths
from google.colab import drive, files
import os
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Drive already mounted')

import sys, shutil, json, zipfile, logging
from pathlib import Path

# Pipeline source code on Drive
PIPELINE_ROOT = Path('/content/drive/MyDrive/spink7_pipeline')
assert PIPELINE_ROOT.exists(), f'Pipeline root not found: {PIPELINE_ROOT}. Upload src/ to Google Drive.'
sys.path.insert(0, str(PIPELINE_ROOT))

# Experiment output directory on Drive (persistent)
EXP_ID = 'EXP-04-dry-run'
DRIVE_OUTPUT = Path(f'/content/drive/MyDrive/v3_gpu_results/{EXP_ID}')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

# Local working directory (fast SSD)
WORK_DIR = Path(f'/content/{EXP_ID}_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s %(message)s')
logger = logging.getLogger(EXP_ID)
logger.info('Workspace initialized')

# ── Console log capture ──────────────────────────────────────
import io as _io
_log_buf = _io.StringIO()
_orig_stdout_write = sys.stdout.write
_orig_stderr_write = sys.stderr.write
def _stdout_log_write(data):
    _orig_stdout_write(data)
    _log_buf.write(data)
def _stderr_log_write(data):
    _orig_stderr_write(data)
    _log_buf.write(data)
sys.stdout.write = _stdout_log_write
sys.stderr.write = _stderr_log_write
# ─────────────────────────────────────────────────────────────

Mounted at /content/drive


In [4]:
# Checkpoint & auto-save utilities
import json, shutil, threading, time as _time
from pathlib import Path

class ExperimentCheckpoint:
    """Drive-backed phase checkpoint for resilient experiment execution."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'experiment_checkpoint.json'
        self.state = self._load()

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.state, f, indent=2, default=str)

    def is_done(self, phase: str) -> bool:
        return phase in self.state['phases']

    def mark_done(self, phase: str, data: dict = None):
        self.state['phases'][phase] = {
            'timestamp': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'data': data or {},
        }
        self.save()

    def get_data(self, phase: str) -> dict:
        return self.state['phases'].get(phase, {}).get('data', {})

    def summary(self):
        n = len(self.state['phases'])
        print(f'Checkpoint: {self.exp_id} — {n} phase(s) completed')
        for phase, info in self.state['phases'].items():
            print(f'  ✓ {phase} ({info["timestamp"]})')


class TimingLog:
    """Wall-clock timing log for each experiment phase, persisted to Drive."""

    def __init__(self, drive_output: Path, exp_id: str):
        self.drive_output = drive_output
        self.exp_id = exp_id
        self.path = drive_output / 'timing_log.json'
        self.log = self._load()
        self._active = {}

    def _load(self) -> dict:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return {'exp_id': self.exp_id, 'phases': {}, 'gpu_info': None}

    def save(self):
        self.drive_output.mkdir(parents=True, exist_ok=True)
        with open(self.path, 'w') as f:
            json.dump(self.log, f, indent=2, default=str)

    def record_gpu(self, gpu_name: str, gpu_memory_gb: float = None):
        self.log['gpu_info'] = {
            'name': gpu_name,
            'memory_gb': gpu_memory_gb,
            'recorded_at': _time.strftime('%Y-%m-%d %H:%M:%S'),
        }
        self.save()

    def start(self, phase: str):
        self._active[phase] = _time.time()
        print(f'⏱ {phase} started at {_time.strftime("%H:%M:%S")}')

    def stop(self, phase: str, extra: dict = None):
        t_start = self._active.pop(phase, _time.time())
        elapsed = _time.time() - t_start
        entry = {
            'start': _time.strftime('%Y-%m-%d %H:%M:%S', _time.localtime(t_start)),
            'end': _time.strftime('%Y-%m-%d %H:%M:%S'),
            'wall_seconds': round(elapsed, 1),
            'wall_human': f'{elapsed/3600:.2f} h' if elapsed > 3600 else f'{elapsed/60:.1f} min',
        }
        if extra:
            entry.update(extra)
        self.log['phases'][phase] = entry
        self.save()
        print(f'⏱ {phase} completed in {entry["wall_human"]}')

    def summary(self):
        total = sum(p.get('wall_seconds', 0) for p in self.log['phases'].values())
        gpu = self.log.get('gpu_info')
        gpu_str = f' on {gpu["name"]}' if gpu else ''
        print(f'Timing: {self.exp_id}{gpu_str} — {len(self.log["phases"])} phase(s), total {total/3600:.2f} h')
        for name, info in self.log['phases'].items():
            print(f'  {name}: {info.get("wall_human", "?")}')


def start_periodic_sync(work_dir: Path, drive_output: Path, interval_s: int = 300):
    """Background thread that syncs checkpoint/manifest files to Drive every N seconds."""
    stop_event = threading.Event()
    sync_patterns = ['*.chk', '*.json', '*manifest*', '*.npz', '*.npy']
    def _sync():
        while not stop_event.is_set():
            stop_event.wait(interval_s)
            if stop_event.is_set():
                break
            for pat in sync_patterns:
                for f in work_dir.rglob(pat):
                    if f.is_file() and f.stat().st_size < 500_000_000:  # skip files > 500 MB
                        dest = drive_output / f.relative_to(work_dir)
                        dest.parent.mkdir(parents=True, exist_ok=True)
                        try:
                            shutil.copy2(f, dest)
                        except Exception:
                            pass
    t = threading.Thread(target=_sync, daemon=True, name='drive-sync')
    t.start()
    return stop_event

def restore_from_drive(drive_output: Path, work_dir: Path):
    """On session restart, copy checkpoint/manifest files from Drive back to local."""
    restored = 0
    for pat in ['*.chk', '*manifest*', '*.json']:
        for f in drive_output.rglob(pat):
            if f.is_file():
                dest = work_dir / f.relative_to(drive_output)
                dest.parent.mkdir(parents=True, exist_ok=True)
                if not dest.exists():
                    shutil.copy2(f, dest)
                    restored += 1
    if restored:
        print(f'Restored {restored} checkpoint/manifest files from Drive')

# Initialize
ckpt = ExperimentCheckpoint(DRIVE_OUTPUT, EXP_ID)
timing = TimingLog(DRIVE_OUTPUT, EXP_ID)
restore_from_drive(DRIVE_OUTPUT, WORK_DIR)
sync_stop = start_periodic_sync(WORK_DIR, DRIVE_OUTPUT, interval_s=300)
ckpt.summary()
timing.summary()
print('Auto-save: checkpoint/manifest files sync to Drive every 5 min')

Checkpoint: EXP-04-dry-run — 0 phase(s) completed
Timing: EXP-04-dry-run — 0 phase(s), total 0.00 h
Auto-save: checkpoint/manifest files sync to Drive every 5 min


In [5]:
# Cell 3: Verify GPU availability
import openmm
from openmm import unit, Platform

print(f'OpenMM version: {openmm.__version__}')
print(f'Available platforms:')
for i in range(Platform.getNumPlatforms()):
    p = Platform.getPlatform(i)
    print(f'  {p.getName()}: speed={p.getSpeed()}')

# Select best GPU platform: CUDA > OpenCL > CPU
GPU_PLATFORM = None
for name in ['CUDA', 'OpenCL']:
    try:
        plat = Platform.getPlatformByName(name)
        GPU_PLATFORM = name
        print(f'\nUsing {name} platform (speed={plat.getSpeed()})')
        break
    except Exception:
        continue
if GPU_PLATFORM is None:
    raise RuntimeError('No GPU platform available — need CUDA or OpenCL')
print('GPU verification PASSED')

# Record GPU info in timing log
import subprocess
try:
    _gpu_result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        capture_output=True, text=True, timeout=10)
    _gpu_parts = _gpu_result.stdout.strip().split(', ')
    _gpu_name = _gpu_parts[0] if _gpu_parts else 'Unknown'
    _gpu_mem = float(_gpu_parts[1]) / 1024 if len(_gpu_parts) > 1 else None
    timing.record_gpu(_gpu_name, _gpu_mem)
    print(f'GPU recorded: {_gpu_name} ({_gpu_mem:.0f} GB)' if _gpu_mem else f'GPU recorded: {_gpu_name}')
except Exception as e:
    timing.record_gpu('Unknown (nvidia-smi failed)')
    print(f'GPU info: nvidia-smi failed ({e})')

OpenMM version: 8.5
Available platforms:
  Reference: speed=1.0
  CPU: speed=10.0
  OpenCL: speed=50.0

Using OpenCL platform (speed=50.0)
GPU verification PASSED
GPU recorded: NVIDIA A100-SXM4-80GB (80 GB)


In [6]:
# Cell 4: Import pipeline modules
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.config import (
    SystemConfig, MinimizationConfig, EquilibrationConfig,
    ProductionConfig, SMDConfig, UmbrellaConfig, WHAMConfig, KCAL_TO_KJ
)
from src.prep.pdb_fetch import fetch_pdb
from src.prep.pdb_clean import clean_structure
from src.prep.topology import build_topology
from src.prep.solvate import solvate_system
from src.simulate.minimizer import minimize_energy
from src.simulate.equilibrate import run_nvt, run_npt
from src.simulate.production import run_production
from src.simulate.smd import run_smd_campaign
from src.simulate.umbrella import (
    run_umbrella_campaign, generate_window_centers,
    compute_overlap_matrix, diagnose_histogram_coverage
)
from src.simulate.platform import select_platform
from src.analyze.wham import solve_wham, bootstrap_pmf_uncertainty
from src.analyze.jarzynski import jarzynski_free_energy, diagnose_dissipation
from src.analyze.convergence import block_average, autocorrelation_time

print('All pipeline modules imported successfully')

All pipeline modules imported successfully


In [7]:
# Cell 5: Configuration
PDB_ID = '2PTC'
CHAIN_TRYPSIN = 'E'
CHAIN_BPTI = 'I'
TEMPERATURE_K = 310.0

system_config = SystemConfig()
min_config = MinimizationConfig(max_iterations=10000)
eq_config = EquilibrationConfig(
    nvt_duration_ps=10.0,   # DRY RUN: 10 ps
    npt_duration_ps=10.0,   # DRY RUN: 10 ps
    temperature_k=TEMPERATURE_K,
)
prod_config = ProductionConfig(
    duration_ns=0.01,       # DRY RUN: 10 ps
    temperature_k=TEMPERATURE_K,
)
smd_config = SMDConfig(
    spring_constant_kj_mol_nm2=1000.0,
    pulling_velocity_nm_per_ps=0.01,   # DRY RUN: 10× faster
    pull_distance_nm=0.1,              # DRY RUN: 0.1 nm
    n_replicates=2,                    # DRY RUN: 2 reps
)
# NOTE: umbrella_config will be overridden dynamically after equilibration
# to center windows on the actual COM distance.
umbrella_config = UmbrellaConfig(
    xi_min_nm=2.0,                     # DRY RUN: placeholder
    xi_max_nm=2.2,                     # DRY RUN: narrow range
    window_spacing_nm=0.05,
    spring_constant_kj_mol_nm2=1000.0,
    per_window_duration_ns=0.001,      # DRY RUN: 1 ps/window
)
wham_config = WHAMConfig(
    tolerance=1e-4,         # DRY RUN: relaxed
    max_iterations=10000,
    n_bootstrap=5,          # DRY RUN: 5 bootstrap
    histogram_bins=5,       # DRY RUN: match sparse data
)

# Output subdirectories
PREP_DIR = WORK_DIR / 'prep'
SIM_DIR = WORK_DIR / 'simulation'
SMD_DIR = WORK_DIR / 'smd'
US_DIR = WORK_DIR / 'umbrella'
ANALYSIS_DIR = WORK_DIR / 'analysis'
FIGURES_DIR = WORK_DIR / 'figures'
for d in [PREP_DIR, SIM_DIR, SMD_DIR, US_DIR, ANALYSIS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Window centers: {len(generate_window_centers(umbrella_config))} windows')
print('Configuration complete')

Window centers: 5 windows
Configuration complete


In [8]:
# Cell 6: Fetch and clean PDB
raw_pdb_path = fetch_pdb(PDB_ID, output_dir=PREP_DIR)
print(f'Fetched PDB: {raw_pdb_path}')

cleaned_pdb_path = clean_structure(
    raw_pdb_path,
    chains_to_keep={CHAIN_TRYPSIN, CHAIN_BPTI},
    remove_heteroatoms=True,
    remove_waters=True,
)
print(f'Cleaned PDB: {cleaned_pdb_path}')

# Verify structure
with open(cleaned_pdb_path) as f:
    lines = [l for l in f if l.startswith(('ATOM', 'HETATM'))]
    chains_found = set(l[21] for l in lines)
print(f'Chains in cleaned PDB: {chains_found}')
print(f'Total atoms: {len(lines)}')
assert {CHAIN_TRYPSIN, CHAIN_BPTI}.issubset(chains_found), 'Missing required chains'

Fetched PDB: /content/EXP-04-dry-run_work/prep/2PTC.pdb
Cleaned PDB: /content/EXP-04-dry-run_work/prep/prepared/2PTC_cleaned.pdb
Chains in cleaned PDB: {'E', 'I'}
Total atoms: 2083


In [9]:
# Cell 7: Build topology and solvate
from openmm.app import PME

topology, system, modeller = build_topology(
    cleaned_pdb_path, system_config,
)
print(f'Topology: {topology.getNumAtoms()} atoms')

# Identify pull groups (chain-based COM groups)
pull_group_1 = []  # trypsin
pull_group_2 = []  # BPTI
for chain in topology.chains():
    indices = [atom.index for atom in chain.atoms()]
    if chain.id == CHAIN_TRYPSIN:
        pull_group_1 = indices
    elif chain.id == CHAIN_BPTI:
        pull_group_2 = indices

print(f'Pull group 1 (trypsin): {len(pull_group_1)} atoms')
print(f'Pull group 2 (BPTI): {len(pull_group_2)} atoms')
assert len(pull_group_1) > 0 and len(pull_group_2) > 0, 'Pull groups empty'

# Solvate
modeller, n_water, n_pos, n_neg = solvate_system(modeller, system_config)
print(f'Solvated: {n_water} waters, {n_pos} Na+, {n_neg} Cl-')
print(f'Total solvated atoms: {modeller.topology.getNumAtoms()}')

/tmp/ipykernel_1726/2270362098.py:4: UserWarning: build_topology() created a system with 4112 atoms using NoCutoff nonbonded method. For systems of this size, NoCutoff produces physically inaccurate long-range electrostatics. Ensure the system is re-parameterized with PME after solvation, or pass nonbonded_method=PME if a periodic box is already defined.
  topology, system, modeller = build_topology(


Topology: 4112 atoms
Pull group 1 (trypsin): 3220 atoms
Pull group 2 (BPTI): 892 atoms
Solvated: 15380 waters, 42 Na+, 54 Cl-
Total solvated atoms: 50348


In [10]:
# Cell 8: Create simulation, minimize, and save system XML
from openmm.app import ForceField, Simulation, HBonds
from openmm import LangevinMiddleIntegrator, XmlSerializer

# Re-create system with PME for solvated box
ff = ForceField(system_config.force_field, system_config.water_model)
system = ff.createSystem(
    modeller.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1.0 * unit.nanometer,
    constraints=HBonds,
    rigidWater=True,
)

integrator = LangevinMiddleIntegrator(
    TEMPERATURE_K * unit.kelvin,
    1.0 / unit.picosecond,
    0.002 * unit.picoseconds,
)
platform = select_platform(GPU_PLATFORM)
simulation = Simulation(modeller.topology, system, integrator, platform)
simulation.context.setPositions(modeller.positions)
print(f'Platform: {simulation.context.getPlatform().getName()}')

# Minimize
timing.start('minimization')
min_result = minimize_energy(simulation, min_config)
timing.stop('minimization')
print(f'Minimization: {min_result["initial_energy_kj_mol"]:.1f} -> {min_result["final_energy_kj_mol"]:.1f} kJ/mol')
print(f'Converged: {min_result["converged"]}')

# Save system XML for SMD/umbrella campaigns
system_xml_path = SIM_DIR / 'system.xml'
system_xml_path.write_text(XmlSerializer.serialize(system), encoding='utf-8')
print(f'System XML saved: {system_xml_path}')

Platform: OpenCL
⏱ minimization started at 21:51:05


/tmp/ipykernel_1726/2299503335.py:27: UserWarning: Energy minimization did not converge: max force 2439.2595 kJ/mol/nm exceeds tolerance 10.0000 kJ/mol/nm after 10000 iterations.
  min_result = minimize_energy(simulation, min_config)


⏱ minimization completed in 0.3 min
Minimization: -409662.5 -> -834432.3 kJ/mol
Converged: False
System XML saved: /content/EXP-04-dry-run_work/simulation/system.xml


In [11]:
# Cell 9: NVT equilibration
timing.start('nvt')
nvt_result = run_nvt(simulation, eq_config, SIM_DIR / 'nvt')
timing.stop('nvt')
print(f'NVT avg temperature: {nvt_result["avg_temperature_k"]:.2f} K')
print(f'NVT trajectory: {nvt_result["trajectory_path"]}')

⏱ nvt started at 21:51:36
⏱ nvt completed in 0.1 min
NVT avg temperature: 308.24 K
NVT trajectory: /content/EXP-04-dry-run_work/simulation/nvt/nvt_equilibration.dcd


In [12]:
# Cell 10: NPT equilibration
timing.start('npt')
npt_result = run_npt(simulation, eq_config, SIM_DIR / 'npt')
timing.stop('npt')
print(f'NPT avg temperature: {npt_result["avg_temperature_k"]:.2f} K')
print(f'NPT avg density: {npt_result["avg_density_g_cm3"]:.4f} g/cm3')
print(f'NPT topology ref: {npt_result["topology_path"]}')
print(f'Finite-size correction: {npt_result["finite_size_correction_kj_mol"]:.4f} kJ/mol')

# Save equilibrated state for SMD/umbrella
eq_state_path = SIM_DIR / 'npt' / 'npt_final_state.xml'
topology_pdb_path = npt_result['topology_path']
print(f'Equilibrated state: {eq_state_path}')

⏱ npt started at 21:51:49
⏱ npt completed in 0.2 min
NPT avg temperature: 309.89 K
NPT avg density: 1.0094 g/cm3
NPT topology ref: /content/EXP-04-dry-run_work/simulation/npt/topology_reference.pdb
Finite-size correction: 44.4165 kJ/mol
Equilibrated state: /content/EXP-04-dry-run_work/simulation/npt/npt_final_state.xml


In [13]:
# Cell 11: Production MD (100 ns) — with checkpoint/resume support
from src.simulate.production import resume_production

prod_output = SIM_DIR / 'production'
prod_output.mkdir(parents=True, exist_ok=True)

if ckpt.is_done('production'):
    print('⏭ Production already completed, skipping')
else:
    timing.start('production')
    # DRY RUN: catch PhysicalValidityError (NVE drift check on short sims)
    from src import PhysicalValidityError as _PVE
    try:
        # Check for existing checkpoints (from prior interrupted session)
        existing_chk = sorted(prod_output.glob('*.chk'))
        if existing_chk:
            print(f'Resuming production from checkpoint: {existing_chk[-1].name}')
            prod_result = resume_production(simulation, prod_config, prod_output)
        else:
            prod_result = run_production(simulation, prod_config, prod_output)
    except _PVE as e:
        print(f'⚠ PhysicalValidityError (tolerated in dry run): {e}')
        prod_result = {
            'n_frames': 1,
            'total_time_ns': prod_config.duration_ns,
            'trajectory_path': str(prod_output / 'production.dcd'),
            'topology_path': str(topology_pdb_path),
        }
    timing.stop('production', {'n_frames': prod_result['n_frames'], 'total_time_ns': prod_result['total_time_ns']})

    print(f'Production: {prod_result["n_frames"]} frames, {prod_result["total_time_ns"]:.1f} ns')
    print(f'Trajectory: {prod_result["trajectory_path"]}')

    # Backup key files to Drive immediately
    for src_file in [system_xml_path, eq_state_path, Path(topology_pdb_path)]:
        if src_file.exists():
            shutil.copy2(src_file, DRIVE_OUTPUT / src_file.name)
    ckpt.mark_done('production', {
        'n_frames': prod_result['n_frames'],
        'total_time_ns': prod_result['total_time_ns'],
    })
    logger.info('Production complete; backed up to Drive')

⏱ production started at 21:52:12


ERROR:src.simulate.production:IV-5 violated: NVE reference energy drift 0.120326 kJ/mol/ns/atom


⚠ PhysicalValidityError (tolerated in dry run): IV-5 violated: total energy drift must remain below 0.1 kJ/mol/ns/atom during production
⏱ production completed in 0.2 min
Production: 1 frames, 0.0 ns
Trajectory: /content/EXP-04-dry-run_work/simulation/production/production.dcd


In [14]:
# Cell 12: SMD campaign — with checkpoint guard
if ckpt.is_done('smd'):
    print('⏭ SMD already completed, loading saved work values')
    _smd_npz = DRIVE_OUTPUT / 'analysis' / 'pmf_data.npz'
    if _smd_npz.exists():
        work_values = np.load(str(_smd_npz))['work_values_kj_mol']
    else:
        work_values = np.load(str(WORK_DIR / 'analysis' / 'pmf_data.npz'))['work_values_kj_mol']
    print(f'Loaded {len(work_values)} work values')
else:
    timing.start('smd')
    smd_results = run_smd_campaign(
        equilibrated_state_path=eq_state_path,
        system_xml_path=system_xml_path,
        config=smd_config,
        pull_group_1=pull_group_1,
        pull_group_2=pull_group_2,
        output_dir=SMD_DIR,
        topology_pdb_path=Path(topology_pdb_path),
        platform_name=GPU_PLATFORM,
        n_workers=1,
    )
    timing.stop('smd', {'n_replicates': len(smd_results)})
    print(f'SMD: {len(smd_results)} replicates completed')
    work_values = np.array([r['total_work_kj_mol'] for r in smd_results])
    print(f'Work: mean={np.mean(work_values):.2f}, std={np.std(work_values):.2f} kJ/mol')

    # Save work values immediately for checkpoint recovery
    np.savez(ANALYSIS_DIR / 'smd_work_values.npz', work_values_kj_mol=work_values)
    shutil.copy2(ANALYSIS_DIR / 'smd_work_values.npz', DRIVE_OUTPUT / 'smd_work_values.npz')
    ckpt.mark_done('smd', {'n_replicates': len(smd_results)})

⏱ smd started at 21:52:30


⏱ smd completed in 0.4 min
SMD: 2 replicates completed
Work: mean=-30.19, std=2.03 kJ/mol


In [15]:
# Cell 13: Jarzynski free energy from SMD
jarz_result = jarzynski_free_energy(work_values, TEMPERATURE_K)
diss_result = diagnose_dissipation(work_values, TEMPERATURE_K)

print(f'Jarzynski \u0394G = {jarz_result["delta_g_kcal_mol"]:.2f} kcal/mol')
print(f'Cumulant-2 \u0394G = {jarz_result["delta_g_cumulant2_kcal_mol"]:.2f} kcal/mol')
print(f'Mean work = {jarz_result["mean_work_kj_mol"]:.2f} kJ/mol')
print(f'W_diss/kBT = {diss_result["w_diss_over_kbt"]:.2f}')
print(f'Efficiency ratio = {diss_result["efficiency_ratio"]:.4f}')

Jarzynski ΔG = -7.39 kcal/mol
Cumulant-2 ΔG = -7.41 kcal/mol
Mean work = -30.19 kJ/mol
W_diss/kBT = 0.28
Efficiency ratio = 0.7546


/tmp/ipykernel_1726/3238606594.py:2: UserWarning: Work distribution failed the Anderson-Darling normality test. The second-order cumulant expansion is unreliable; use the BAR estimator (bar_free_energy) for robust results.
  jarz_result = jarzynski_free_energy(work_values, TEMPERATURE_K)


In [16]:
# DRY RUN: Override umbrella config — center windows on actual COM distance
import numpy as _np_us
from openmm import unit as _unit_us
_eq_pos = simulation.context.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(_unit_us.nanometer)
_com1 = _np_us.mean(_eq_pos[pull_group_1], axis=0)
_com2 = _np_us.mean(_eq_pos[pull_group_2], axis=0)
_xi0 = float(_np_us.linalg.norm(_com1 - _com2))
print(f'Equilibrium COM distance: {_xi0:.3f} nm')
umbrella_config = UmbrellaConfig(
    xi_min_nm=round(_xi0 - 0.04, 3),
    xi_max_nm=round(_xi0 + 0.04, 3),
    window_spacing_nm=0.02,
    spring_constant_kj_mol_nm2=1000.0,
    per_window_duration_ns=0.001,
)
print(f'DRY RUN umbrella: {len(generate_window_centers(umbrella_config))} windows around {_xi0:.2f} nm')

# Cell 14: Umbrella sampling campaign — manifest auto-resume is built in
if ckpt.is_done('umbrella'):
    print('⏭ Umbrella sampling already completed, loading saved timeseries')
    _us_npz = DRIVE_OUTPUT / 'umbrella_timeseries.npz'
    if not _us_npz.exists():
        _us_npz = ANALYSIS_DIR / 'umbrella_timeseries.npz'
    _data = np.load(str(_us_npz))
    window_centers = _data['window_centers']
    xi_timeseries_list = [_data[f'window_{i}'] for i in range(len(window_centers))]
    us_results = [{'window_center_nm': float(window_centers[i]),
                   'xi_timeseries': xi_timeseries_list[i]}
                  for i in range(len(window_centers))]
    print(f'Loaded {len(us_results)} umbrella windows')
else:
    timing.start('umbrella')
    us_results = run_umbrella_campaign(
        equilibrated_state_path=eq_state_path,
        system_xml_path=system_xml_path,
        config=umbrella_config,
        pull_group_1=pull_group_1,
        pull_group_2=pull_group_2,
        output_dir=US_DIR,
        topology_pdb_path=Path(topology_pdb_path),
        platform_name=GPU_PLATFORM,
        n_workers=1,
        strict_coverage_check=False,  # DRY RUN: sparse data
    )
    timing.stop('umbrella', {'n_windows': len(us_results)})
    print(f'Umbrella: {len(us_results)} windows completed')
    for r in us_results[:3]:
        print(f'  Window {r["window_id"]}: center={r["window_center_nm"]:.3f}, mean_xi={r["mean_xi_nm"]:.3f} nm')

    # Save timeseries immediately
    xi_timeseries_list = [r['xi_timeseries'] for r in us_results]
    window_centers = np.array([r['window_center_nm'] for r in us_results])
    np.savez(
        ANALYSIS_DIR / 'umbrella_timeseries.npz',
        **{f'window_{i}': ts for i, ts in enumerate(xi_timeseries_list)},
        window_centers=window_centers,
    )
    shutil.copy2(ANALYSIS_DIR / 'umbrella_timeseries.npz', DRIVE_OUTPUT / 'umbrella_timeseries.npz')
    ckpt.mark_done('umbrella', {'n_windows': len(us_results)})

Equilibrium COM distance: 2.575 nm
DRY RUN umbrella: 5 windows around 2.57 nm
⏱ umbrella started at 21:53:05


⏱ umbrella completed in 5.9 min
Umbrella: 5 windows completed
  Window 1: center=2.535, mean_xi=2.899 nm
  Window 2: center=2.555, mean_xi=2.822 nm
  Window 3: center=2.575, mean_xi=2.940 nm


In [18]:
# Cell 15: WHAM PMF reconstruction
xi_timeseries_list = [r['xi_timeseries'] for r in us_results]
window_centers = np.array([r['window_center_nm'] for r in us_results])
spring_constants = np.full(len(us_results), umbrella_config.spring_constant_kj_mol_nm2)

# Overlap diagnostics
overlap_matrix = compute_overlap_matrix(xi_timeseries_list)
min_adjacent_overlap = min(
    overlap_matrix[i, i+1] for i in range(len(us_results) - 1)
)
print(f'Min adjacent overlap: {min_adjacent_overlap:.3f} (IV-8 requires >= 0.10)')

coverage = diagnose_histogram_coverage(xi_timeseries_list, window_centers)
print(f'Coverage fraction: {coverage["coverage_fraction"]:.3f}')
print(f'Coverage holes: {coverage["has_coverage_holes"]}')

# Solve WHAM — DRY RUN: tolerate coverage holes from sparse data
from src import PhysicalValidityError as _PVE_WHAM
timing.start('wham')
try:
    wham_result = solve_wham(
        xi_timeseries_list, window_centers, spring_constants,
        TEMPERATURE_K, wham_config,
    )
    print(f'WHAM converged: {wham_result["converged"]} in {wham_result["n_iterations"]} iterations')
except _PVE_WHAM as e:
    print(f'⚠ WHAM coverage error (tolerated in dry run): {e}')
    # Create synthetic WHAM result for downstream cells
    _xi = np.linspace(float(window_centers[0]), float(window_centers[-1]), 20)
    _pmf = np.zeros_like(_xi)
    _pmf[0] = -10.0 * KCAL_TO_KJ  # synthetic bound-state minimum
    wham_result = {
        'converged': False,
        'n_iterations': 0,
        'pmf_kj_mol': _pmf,
        'pmf_kcal_mol': _pmf / KCAL_TO_KJ,
        'xi_bins': _xi,
    }
timing.stop('wham', {'converged': wham_result['converged'], 'n_iterations': wham_result['n_iterations']})

Min adjacent overlap: 0.000 (IV-8 requires >= 0.10)
Coverage fraction: 0.050
Coverage holes: True
⏱ wham started at 21:59:26
⚠ WHAM coverage error (tolerated in dry run): WHAM input fails coverage requirement: 40% of bins have zero counts. Add umbrella windows to fill gaps.
⏱ wham completed in 0.0 min


/tmp/ipykernel_1726/3795040974.py:21: UserWarning: WHAM input has 2/5 bins with zero counts (coverage gaps near xi = [2.85648822 2.88104128] nm). PMF will be infinite at these bins.
  wham_result = solve_wham(


In [19]:
# Cell 16: Bootstrap PMF uncertainty
# DRY RUN: bootstrap may fail on sparse data — provide fallback
timing.start('bootstrap')
try:
    bootstrap_result = bootstrap_pmf_uncertainty(
        xi_timeseries_list, window_centers, spring_constants,
        TEMPERATURE_K, wham_config,
    )
except Exception as _boot_err:
    print(f'⚠ Bootstrap failed (tolerated in dry run): {_boot_err}')
    bootstrap_result = {
        'pmf_mean': wham_result['pmf_kj_mol'],
        'pmf_std': np.ones_like(wham_result['pmf_kj_mol']),
        'xi_bins': wham_result['xi_bins'],
        'n_eff_per_window': np.ones(len(window_centers)),
    }
timing.stop('bootstrap', {'n_bootstrap': wham_config.n_bootstrap})

pmf_mean = bootstrap_result['pmf_mean']
pmf_std = bootstrap_result['pmf_std']
xi_bins = bootstrap_result['xi_bins']

print(f'PMF shape: {pmf_mean.shape}')
print(f'Mean N_eff per window: {np.mean(bootstrap_result["n_eff_per_window"]):.0f}')

⏱ bootstrap started at 21:59:32
⏱ bootstrap completed in 0.0 min
PMF shape: (5,)
Mean N_eff per window: 1


In [20]:
# Cell 17: Extract binding free energy
pmf_kj = wham_result['pmf_kj_mol']
pmf_kcal = wham_result['pmf_kcal_mol']
xi = wham_result['xi_bins']

# Find the bound-state minimum
finite_mask = np.isfinite(pmf_kcal)
bound_region = (xi < 2.5) & finite_mask
if np.any(bound_region):
    bound_min_idx = np.argmin(pmf_kcal[bound_region])
    bound_min_val = pmf_kcal[bound_region][bound_min_idx]
    bound_min_xi = xi[bound_region][bound_min_idx]
else:
    bound_min_idx = np.argmin(pmf_kcal[finite_mask])
    bound_min_val = pmf_kcal[finite_mask][bound_min_idx]
    bound_min_xi = xi[finite_mask][bound_min_idx]

# The PMF is shifted so the last bin = 0 (dissociated reference)
# DG_bind = PMF(bound minimum) - PMF(dissociated) = bound_min_val - 0
dg_bind_kcal = float(bound_min_val)
dg_bind_kj = dg_bind_kcal * KCAL_TO_KJ

# Uncertainty from bootstrap at the bound minimum bin
if np.any(bound_region):
    bound_idx_global = np.where(bound_region)[0][bound_min_idx]
else:
    bound_idx_global = np.where(finite_mask)[0][bound_min_idx]
dg_uncertainty_kcal = float(pmf_std[bound_idx_global]) / KCAL_TO_KJ

print(f'\n=== BINDING FREE ENERGY ===')
print(f'\u0394G_bind = {dg_bind_kcal:.2f} \u00b1 {dg_uncertainty_kcal:.2f} kcal/mol')
print(f'\u0394G_bind = {dg_bind_kj:.2f} kJ/mol')
print(f'Bound minimum at \u03be = {bound_min_xi:.3f} nm')

# Success classification
EXPECTED = -18.0
PASS_LO, PASS_HI = -24.7, -11.3
MARGINAL_LO, MARGINAL_HI = -31.4, -4.6

if PASS_LO <= dg_bind_kcal <= PASS_HI:
    verdict = 'PASS'
elif MARGINAL_LO <= dg_bind_kcal <= MARGINAL_HI:
    verdict = 'MARGINAL'
else:
    verdict = 'FAIL'
print(f'Verdict: {verdict} (expected {EXPECTED} kcal/mol, range [{PASS_LO}, {PASS_HI}])')


=== BINDING FREE ENERGY ===
ΔG_bind = -10.00 ± 0.00 kcal/mol
ΔG_bind = -41.84 kJ/mol
Bound minimum at ξ = 2.535 nm
Verdict: MARGINAL (expected -18.0 kcal/mol, range [-24.7, -11.3])


In [22]:
# Cell 18: Generate figures
# Figure 1: PMF profile
fig, ax = plt.subplots(figsize=(10, 6))
finite = np.isfinite(pmf_kcal)
ax.plot(xi[finite], pmf_kcal[finite], 'b-', linewidth=2, label='WHAM PMF')
# Bootstrap arrays may differ in size from WHAM arrays (e.g. dry-run fallback)
_boot_kcal = pmf_mean / KCAL_TO_KJ
_boot_std_kcal = pmf_std / KCAL_TO_KJ
_boot_finite = np.isfinite(_boot_kcal)
ax.fill_between(
    xi_bins[_boot_finite],
    (_boot_kcal - _boot_std_kcal)[_boot_finite],
    (_boot_kcal + _boot_std_kcal)[_boot_finite],
    alpha=0.3, color='blue', label='Bootstrap \u00b11\u03c3',
)
ax.axhline(y=EXPECTED, color='red', linestyle='--', label=f'Exp. \u0394G = {EXPECTED} kcal/mol')
ax.axhspan(PASS_LO, PASS_HI, alpha=0.1, color='green', label='PASS range')
ax.set_xlabel('\u03be (nm)', fontsize=14)
ax.set_ylabel('PMF (kcal/mol)', fontsize=14)
ax.set_title('EXP-04: BPTI\u2013Trypsin PMF', fontsize=16)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'pmf_profile.png', dpi=150)
plt.close(fig)
print('PMF profile saved')

# Figure 2: SMD work distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(work_values, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(jarz_result['delta_g_kj_mol'], color='red', linewidth=2, label=f'Jarzynski \u0394G = {jarz_result["delta_g_kcal_mol"]:.1f} kcal/mol')
ax.axvline(np.mean(work_values), color='green', linewidth=2, linestyle='--', label=f'Mean W = {np.mean(work_values)/KCAL_TO_KJ:.1f} kcal/mol')
ax.set_xlabel('Work (kJ/mol)', fontsize=14)
ax.set_ylabel('Count', fontsize=14)
ax.set_title('EXP-04: SMD Work Distribution (50 replicates)', fontsize=14)
ax.legend(fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'smd_work_distribution.png', dpi=150)
plt.close(fig)
print('SMD work distribution saved')

# Figure 3: Umbrella window overlap heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(overlap_matrix, cmap='viridis', vmin=0, vmax=1, aspect='auto')
ax.set_xlabel('Window index', fontsize=12)
ax.set_ylabel('Window index', fontsize=12)
ax.set_title('EXP-04: Umbrella Window Overlap Matrix', fontsize=14)
fig.colorbar(im, ax=ax, label='Overlap fraction')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'overlap_matrix.png', dpi=150)
plt.close(fig)
print('Overlap matrix saved')

PMF profile saved
SMD work distribution saved
Overlap matrix saved


In [23]:
# Cell 19: Save results JSON
results = {
    'experiment_id': EXP_ID,
    'pdb_id': PDB_ID,
    'temperature_k': TEMPERATURE_K,
    'dg_bind_kcal_mol': dg_bind_kcal,
    'dg_bind_kj_mol': dg_bind_kj,
    'dg_uncertainty_kcal_mol': dg_uncertainty_kcal,
    'bound_minimum_xi_nm': float(bound_min_xi),
    'verdict': verdict,
    'expected_kcal_mol': EXPECTED,
    'pass_range_kcal_mol': [PASS_LO, PASS_HI],
    'jarzynski_dg_kcal_mol': jarz_result['delta_g_kcal_mol'],
    'jarzynski_cumulant2_kcal_mol': jarz_result['delta_g_cumulant2_kcal_mol'],
    'mean_work_kj_mol': jarz_result['mean_work_kj_mol'],
    'w_diss_over_kbt': diss_result['w_diss_over_kbt'],
    'wham_converged': wham_result['converged'],
    'wham_iterations': wham_result['n_iterations'],
    'min_adjacent_overlap': float(min_adjacent_overlap),
    'coverage_fraction': float(coverage['coverage_fraction']),
    'n_smd_replicates': len(smd_results),
    'n_umbrella_windows': len(us_results),
    'minimization': min_result,
    'nvt_avg_temp_k': nvt_result['avg_temperature_k'],
    'npt_avg_density': npt_result['avg_density_g_cm3'],
    'production_ns': prod_result['total_time_ns'],
    'finite_size_correction_kj_mol': npt_result['finite_size_correction_kj_mol'],
    'timing': timing.log['phases'],
    'gpu_info': timing.log.get('gpu_info'),
}

results_path = WORK_DIR / 'results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Results saved: {results_path}')

# Also save PMF data as numpy arrays
np.savez(
    ANALYSIS_DIR / 'pmf_data.npz',
    xi_bins=xi, pmf_kj_mol=pmf_kj, pmf_kcal_mol=pmf_kcal,
    pmf_mean=pmf_mean, pmf_std=pmf_std,
    xi_bins_bootstrap=xi_bins,
    work_values_kj_mol=work_values,
)
print('PMF data saved as npz')

# Save umbrella timeseries for downstream experiments (EXP-09, EXP-10)
np.savez(
    ANALYSIS_DIR / 'umbrella_timeseries.npz',
    **{f'window_{i}': ts for i, ts in enumerate(xi_timeseries_list)},
    window_centers=window_centers,
    spring_constants=spring_constants,
)
print('Umbrella timeseries saved for downstream experiments')

# Print timing summary
timing.summary()

Results saved: /content/EXP-04-dry-run_work/results.json
PMF data saved as npz
Umbrella timeseries saved for downstream experiments
Timing: EXP-04-dry-run on NVIDIA A100-SXM4-80GB — 8 phase(s), total 0.12 h
  minimization: 0.3 min
  nvt: 0.1 min
  npt: 0.2 min
  production: 0.2 min
  smd: 0.4 min
  umbrella: 5.9 min
  wham: 0.0 min
  bootstrap: 0.0 min


In [24]:
# Cell 20: Copy all results to Drive, zip, and auto-download

# Stop periodic sync before final copy
sync_stop.set()

# Copy working directory to Drive
for item in WORK_DIR.rglob('*'):
    if item.is_file():
        rel_path = item.relative_to(WORK_DIR)
        dest = DRIVE_OUTPUT / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

# Create zip file
# ── Save full console log ────────────────────────────────────
_console_log_text = _log_buf.getvalue()
(WORK_DIR / 'console_log.txt').write_text(_console_log_text)
(DRIVE_OUTPUT / 'console_log.txt').write_text(_console_log_text)
print(f'Console log saved: {len(_console_log_text)} chars')
# ─────────────────────────────────────────────────────────────
zip_path = Path(f'/content/{EXP_ID}_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in WORK_DIR.rglob('*'):
        if item.is_file():
            arcname = f'{EXP_ID}/{item.relative_to(WORK_DIR)}'
            zf.write(item, arcname)

ckpt.mark_done('complete')
print(f'Zip created: {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)')
print(f'Results also saved to Drive: {DRIVE_OUTPUT}')

# Auto-download
files.download(str(zip_path))
print('Download initiated!')

Console log saved: 5780 chars
Zip created: /content/EXP-04-dry-run_results.zip (27.6 MB)
Results also saved to Drive: /content/drive/MyDrive/v3_gpu_results/EXP-04-dry-run


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated!
